# Time Series Analysis of Announced Prefix Visibility

This notebook loads the per-datetime processed visibility files for **announced prefixes** and computes time series metrics.

**Purpose:**
- Load all `visibility_*.pkl` files from the processing step
- Compute time series per ASN and IP version:
  - Total number of prefixes (all visibility levels)
  - Total prefixes with visibility >= 70%
  - Total prefixes with visibility >= 75%
  - Total prefixes with visibility >= 80%
  - Total prefixes with visibility >= 90%
  - Total prefixes with visibility >= 95%
  - Total prefixes with visibility == 100%

**Input:** Per-datetime visibility files from `6-Compute_announced_prefixes_high_visibility.ipynb`

**Output:** Time series data structures for further analysis

## Imports

In [ ]:
import os
import json
import pickle
import datetime
import glob
import pandas as pd
from collections import defaultdict
from pathlib import Path
from multiprocessing import Pool
from tqdm import tqdm

## Configuration

In [ ]:
REPO_ROOT = os.path.abspath("..")
fd = open(os.path.join(REPO_ROOT, "settings.json"))
parameters = json.load(fd)
for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
    if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
        parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))
fd.close()

try:
    data_dir = parameters["DATA_DIR"]
    start_date = parameters["START_DATE"]
    end_date = parameters["END_DATE"]
    collectors = parameters["COLLECTORS"]

    start_date = datetime.datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.datetime.strptime(end_date, "%Y-%m-%d")
    total_rrcs = len(collectors)

    # Input directory with per-datetime files
    input_dir = parameters.get(
        "VISIBILITY_ANNOUNCED_OUTPUT_DIR", f"{data_dir}/processed/announced_visibility/"
    )
except:
    raise ValueError("Invalid parameter file")

print(f"Configuration loaded:")
print(f"  Date range: {start_date.date()} to {end_date.date()}")
print(f"  Total RRCs: {total_rrcs}")
print(f"  Input dir: {input_dir}")

# Use 35% of available CPUs to avoid overloading the server
N_WORKERS = max(1, int(os.cpu_count() * 0.35))
print(f"  Workers: {N_WORKERS} / {os.cpu_count()} CPUs")

## Discover Processed Files

In [ ]:
# Find all visibility files
pattern = f"{input_dir}/visibility_*.pkl"
visibility_files = sorted(glob.glob(pattern))

print(f"Found {len(visibility_files)} processed visibility files")

# Calculate expected number of files
delta = datetime.timedelta(hours=8)
expected_count = 0
current = start_date
while current < end_date:
    expected_count += 1
    current += delta

print(f"Expected: {expected_count} files")

if len(visibility_files) < expected_count:
    missing = expected_count - len(visibility_files)
    print(
        f"⚠️  Warning: {missing} files are missing ({missing/expected_count*100:.1f}%)"
    )
elif len(visibility_files) == expected_count:
    print(f"✓ All expected files are present")
else:
    print(
        f"⚠️  Warning: More files than expected ({len(visibility_files)} > {expected_count})"
    )

## Compute Time Series

Process each file and extract the requested metrics per ASN and IP version.

In [ ]:
def process_file(filepath):
    """Process a single visibility file. Returns list of per-ASN per-IP rows."""
    try:
        with open(filepath, "rb") as f:
            data = pickle.load(f)

        timestamp = data["timestamp"]
        results = []

        for asn, asn_info in data["asn_data"].items():
            if asn_info.get("bogus", False):
                continue
            for ip_version in [4, 6]:
                if ip_version not in asn_info:
                    continue
                prefixes = asn_info[ip_version].get("prefixes", {})
                total = len(prefixes)
                v70 = v75 = v80 = v90 = v95 = v100 = 0
                for prefix_info in prefixes.values():
                    v = prefix_info.get("visibility", 0)
                    # Cascading checks: once a threshold fails, higher ones also fail
                    if v >= 70:
                        v70 += 1
                        if v >= 75:
                            v75 += 1
                            if v >= 80:
                                v80 += 1
                                if v >= 90:
                                    v90 += 1
                                    if v >= 95:
                                        v95 += 1
                                        if v == 100:
                                            v100 += 1
                results.append(
                    (asn, ip_version, timestamp, total, v70, v75, v80, v90, v95, v100)
                )
        return results
    except Exception as e:
        print(f"\u274c Error loading {filepath}: {e}")
        return []


print(f"Processing {len(visibility_files)} files with {N_WORKERS} workers...\n")

timeseries_data = defaultdict(
    lambda: {
        4: {
            "total": {},
            "visibility_70": {},
            "visibility_75": {},
            "visibility_80": {},
            "visibility_90": {},
            "visibility_95": {},
            "visibility_100": {},
        },
        6: {
            "total": {},
            "visibility_70": {},
            "visibility_75": {},
            "visibility_80": {},
            "visibility_90": {},
            "visibility_95": {},
            "visibility_100": {},
        },
    }
)

n_workers = N_WORKERS
failed_count = 0

with Pool(processes=n_workers) as pool:
    for file_results in tqdm(
        pool.imap_unordered(process_file, visibility_files),
        total=len(visibility_files),
    ):
        if not file_results:
            failed_count += 1
            continue
        for asn, ip_version, ts, total, v70, v75, v80, v90, v95, v100 in file_results:
            d = timeseries_data[asn][ip_version]
            d["total"][ts] = total
            d["visibility_70"][ts] = v70
            d["visibility_75"][ts] = v75
            d["visibility_80"][ts] = v80
            d["visibility_90"][ts] = v90
            d["visibility_95"][ts] = v95
            d["visibility_100"][ts] = v100

timeseries_data = dict(timeseries_data)

print(
    f"\n ✅ Done. {len(visibility_files) - failed_count}/{len(visibility_files)} files processed."
)
print(f"  Total ASNs with data: {len(timeseries_data):,}")
if failed_count:
    print(f"🚫   {failed_count} files failed to load")

## Summary Statistics

In [ ]:
print("=" * 60)
print("TIME SERIES SUMMARY")
print("=" * 60)

# Count ASNs with IPv4 vs IPv6 data
asns_with_ipv4 = 0
asns_with_ipv6 = 0

for asn, asn_data in timeseries_data.items():
    if asn_data[4]["total"]:
        asns_with_ipv4 += 1
    if asn_data[6]["total"]:
        asns_with_ipv6 += 1

print(f"\nASN Statistics:")
print(f"  Total ASNs: {len(timeseries_data):,}")
print(f"  ASNs with IPv4 data: {asns_with_ipv4:,}")
print(f"  ASNs with IPv6 data: {asns_with_ipv6:,}")

# Collect all timestamps
all_timestamps = set()
for asn, asn_data in timeseries_data.items():
    for ip_version in [4, 6]:
        all_timestamps.update(asn_data[ip_version]["total"].keys())

all_timestamps = sorted(all_timestamps)

print(f"\nTemporal Coverage:")
print(f"  Total unique timestamps: {len(all_timestamps)}")
if all_timestamps:
    print(f"  Date range: {min(all_timestamps)} to {max(all_timestamps)}")
    duration = max(all_timestamps) - min(all_timestamps)
    print(f"  Duration: {duration.days} days")

print(f"\nMetrics Available per ASN and IP version:")
print(f"  - Total prefixes (all visibility levels)")
print(f"  - Visibility prefixes >= 70%")
print(f"  - Visibility prefixes >= 75%")
print(f"  - Visibility prefixes >= 80%")
print(f"  - Visibility prefixes >= 90%")
print(f"  - Visibility prefixes >= 95%")
print(f"  - Visibility prefixes == 100%")

print("\n" + "=" * 60)
print("✓ Time series data is ready!")
print("=" * 60)

## Convert to Pandas DataFrames

Convert the time series data to pandas DataFrames for easier analysis and visualization.

In [ ]:
def create_dataframe_for_asn(asn, ip_version):
    """
    Create a pandas DataFrame for a specific ASN and IP version.

    Args:
        asn: ASN number
        ip_version: 4 or 6

    Returns:
        DataFrame with columns: timestamp, total, visibility_70, visibility_75, visibility_80, visibility_90, visibility_95, visibility_100
    """
    if asn not in timeseries_data:
        return None

    data = timeseries_data[asn][ip_version]

    if not data["total"]:
        return None

    # Get all timestamps
    timestamps = sorted(data["total"].keys())

    # Build DataFrame
    df = pd.DataFrame(
        {
            "timestamp": timestamps,
            "total": [data["total"][ts] for ts in timestamps],
            "visibility_70": [data["visibility_70"][ts] for ts in timestamps],
            "visibility_75": [data["visibility_75"][ts] for ts in timestamps],
            "visibility_80": [data["visibility_80"][ts] for ts in timestamps],
            "visibility_90": [data["visibility_90"][ts] for ts in timestamps],
            "visibility_95": [data["visibility_95"][ts] for ts in timestamps],
            "visibility_100": [data["visibility_100"][ts] for ts in timestamps],
        }
    )

    df.set_index("timestamp", inplace=True)
    return df


# # Example usage
# if sample_asn:
#     print(f"\nExample DataFrame for ASN {sample_asn}:\n")

#     for ip_version in [4, 6]:
#         df = create_dataframe_for_asn(sample_asn, ip_version)
#         if df is not None:
#             print(f"IPv{ip_version}:")
#             print(df.head())
#             print(f"\nShape: {df.shape}")
#             print(f"\nBasic statistics:")
#             print(df.describe())
#             print("\n" + "=" * 60 + "\n")

## Save Time Series Data

Save the time series data for future use.

In [ ]:
output_dir = f"{data_dir}/processed"
os.makedirs(output_dir, exist_ok=True)

output_file = f"{output_dir}/timeseries_prefix_announced_visibility.pkl"

with open(output_file, "wb") as fd:
    pickle.dump(timeseries_data, fd)

file_size_mb = os.path.getsize(output_file) / (1024**2)
print(f"✓ Saved time series data to: {output_file}")
print(f"  File size: {file_size_mb:.2f} MB")

# Also save metadata about the time series
metadata = {
    "total_asns": len(timeseries_data),
    "asns_with_ipv4": asns_with_ipv4,
    "asns_with_ipv6": asns_with_ipv6,
    "timestamps": all_timestamps,
    "date_range": {
        "start": str(min(all_timestamps)) if all_timestamps else None,
        "end": str(max(all_timestamps)) if all_timestamps else None,
    },
    "metrics": [
        "total",
        "visibility_70",
        "visibility_75",
        "visibility_80",
        "visibility_90",
        "visibility_95",
        "visibility_100",
    ],
    "created": datetime.datetime.now().isoformat(),
}

metadata_file = f"{output_dir}/timeseries_announced_metadata.json"
with open(metadata_file, "w") as fd:
    json.dump(metadata, fd, indent=2, default=str)

print(f"✓ Saved metadata to: {metadata_file}")

print("Memory freed.")